# Lesson 08: Pre-Trained Networks, Transfer Learning & Training Tricks

In this notebook, you will:
1. Load a pre-trained **ResNet-18** (`torchvision.models`) and adapt it via **Transfer Learning**.
2. Apply **BatchNorm**, **Dropout**, and **FGSM Adversarial Perturbations**.
3. Complete the **Lab**: Classify **Oxford-IIIT Pet Breeds** (37 classes) on Colab Free T4 GPU.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

weights = models.ResNet18_Weights.DEFAULT
tfms = weights.transforms()

train_ds = datasets.OxfordIIITPet(root="./data", split="trainval", download=True, transform=tfms)
test_ds = datasets.OxfordIIITPet(root="./data", split="test", download=True, transform=tfms)

train_loader = DataLoader(Subset(train_ds, range(1000)), batch_size=32, shuffle=True)
test_loader = DataLoader(Subset(test_ds, range(400)), batch_size=64)

In [ ]:
model = models.resnet18(weights=weights)
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.fc.in_features, 37)
)
model = model.to(device)

opt = optim.Adam(model.fc.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(2):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
    print(f"Epoch {epoch+1} | Oxford Pets Subset Val Acc: {100 * correct / 400:.2f}%")